In [1]:
import yaml
import numpy as np
import polars as pl

patho_labels  = ['Pathogenic', 'Likely_pathogenic']
benign_labels = ['Benign', 'Likely_benign']

clinvar_labels = patho_labels + benign_labels

# Create input for VEP and annotation pipeline

In [ ]:
CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_20260621.vcf.gz"

!wget -P /s/project/ukbbgym/annotation_files/clinvar/ {CLINVAR_URL}


In [ ]:
annotation_dir = "/s/project/ukbbgym/annotation_files/clinvar"
clinvar_vcf_path = "/s/project/ukbbgym/annotation_files/clinvar/clinvar_20260621.vcf.gz"

clinvar = (
    pl.scan_csv(
        clinvar_vcf_path,
        separator="\t",
        comment_prefix="##",
        schema_overrides={"#CHROM": pl.Utf8, "POS": pl.Int64},
        ignore_errors=True,
    )
    .rename({"#CHROM": "chrom", "POS": "pos", "REF": "ref", "ALT": "alt"})
    .with_columns(
        chrom="chr" + pl.col("chrom").cast(pl.Utf8).str.replace(r"^chr", ""),
        clinical_significance=pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .with_columns(
        id=pl.concat_str(["chrom", "pos", "ref", "alt"], separator=":"),
    )
    .select(["id", "chrom", "pos", "ref", "alt", "clinical_significance"])
    .filter(pl.col("clinical_significance").is_in(clinvar_labels))
    .unique(subset="id")          # one row per variant
    .collect()
)

# 1. variant_metadata.parquet — exactly the schema the Snakefile consumes
clinvar.select("id", "chrom", "pos", "ref", "alt").write_parquet(
    f"{annotation_dir}/variant_metadata.parquet"
)

# 2. keep the labels to join back onto the final annotation output on `id`
clinvar.select("id", "clinical_significance").write_parquet(
    f"{annotation_dir}/clinvar_labels.parquet"
)


# Add more missense annotations

In [2]:
import add_missense_variant_annotations as ann

In [3]:
cv_all = (
    pl.read_parquet("/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet")
)

cv_all

id,chrom,pos,ref,alt,region,gnomade_af,gnomadg_af,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,is_indel,is_insertion,is_deletion,mobi_full_disorder_priority,mobi_curated_disorder_priority,mobi_full_lip_priority,ted_domain,…,gc_is_na,gerpn_is_na,gerps_is_na,grantham_is_na,remapoverlapcl_is_na,remapoverlaptf_is_na,roulette-ar_is_na,roulette-mr_is_na,zoopriphylop_is_na,zoouce_is_na,zooverphylop_is_na,bstatistic_is_na,cdnapos_is_na,dbscsnv-ada_score_is_na,dbscsnv-rf_score_is_na,mamphcons_is_na,mamphylop_is_na,mindisttse_is_na,mindisttss_is_na,mirsvr-aln_is_na,mirsvr-e_is_na,mirsvr-score_is_na,motifdist_is_na,motifecount_is_na,motifehipos_is_na,motifescorechng_is_na,priphcons_is_na,priphylop_is_na,relcdspos_is_na,relprotpos_is_na,relcdnapos_is_na,toverlapmotifs_is_na,targetscan_is_na,verphcons_is_na,verphylop_is_na,cpt1_llr_is_na,blosum62_is_na
str,str,i64,str,str,str,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i8,i8,i8,bool,bool,bool,bool,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""chr4:102657670:A:T""","""chr4""",102657670,"""A""","""T""","""ENSG00000109323""",null,null,3.67,0.335883,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.006,0.000529,0,0,0,0,0,1,-1.7,0.0,0.0,0.0,0,0,0,false,false,false,false,…,0,0,0,1,0,0,1,0,0,1,0,0,1,1,1,0,0,0,0,1,1,1,0,1,1,1,0,0,1,1,1,0,1,0,0,1,1
"""chr4:102657700:G:A""","""chr4""",102657700,"""G""","""A""","""ENSG00000109323""",0.000001,0.000007,9.239,0.898057,0.0,0,0,0.64,0.0,0.01,0.01,0.01,0.003,0.002034,0,0,0,0,0,1,-2.64,0.0,0.0,0.0,0,0,0,false,false,false,true,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102657733:A:G""","""chr4""",102657733,"""A""","""G""","""ENSG00000109323""",null,null,6.017,0.554886,0.0,0,0,0.63,0.0,0.03,0.01,0.03,0.003,0.000405,0,0,0,0,0,1,-3.71,0.0,0.0,0.0,0,0,0,false,false,false,true,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102657793:T:C""","""chr4""",102657793,"""T""","""C""","""ENSG00000109323""",0.000003,null,0.495,-0.19568,0.0,0,0,0.6,0.0,0.02,0.02,0.02,0.003,0.000323,0,0,0,0,0,1,-2.03,0.0,0.0,0.0,0,0,0,false,false,false,true,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102664662:C:T""","""chr4""",102664662,"""C""","""T""","""ENSG00000109323""",0.001235,0.01127,2.691,0.24391,0.0,0,0,0.0,0.0,0.0,0.01,0.0,0.003,0.002195,0,0,0,0,0,1,-1.31,0.0,0.0,0.0,0,0,0,false,false,false,false,…,0,0,0,1,1,1,1,0,0,1,0,0,1,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,1,1,1,1,1,0,0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr1:75761148:T:TG""","""chr1""",75761148,"""T""","""TG""","""ENSG00000117054""",null,null,0.0,0.28051,0.0,1,0,0.77,0.0,0.01,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,1,0,false,false,false,true,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"""chr9:127666254:CTA:C""","""chr9""",127666254,"""CTA""","""C""","""ENSG00000136854""",null,null,0.0,0.28051,0.0,1,0,0.42,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,3,-1.41,0.0,0.0,0.0,1,0,1,false,false,false,true,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"""chr2:47412477:AT:A""","""chr2""",47412477,"""AT""","""A""","""ENSG00000095002""",null,null,0.0,0.28051,0.0,1,0,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,0,1,false,false,false,true,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [ ]:
ann.main(
    "/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet",
    fill_null_defaults_path="fill_null_defaults.yaml",
    output_path="/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_missense.parquet",
    download_dir="/s/project/ukbbgym/annotation_files/more_missense",
    add_alphamissense=False,
    add_cpt1=False,
    add_cadd=False,
    add_gpn_msa=False,
    add_phylop=False,
)

[2026-06-23 23:29:25,678] INFO:add_missense_variant_annotations: === Downloading required files ===


[2026-06-23 23:29:25,747] INFO:add_missense_variant_annotations:   Already complete: /s/project/ukbbgym/annotation_files/more_missense/tmp/clinvar.vcf.gz
[2026-06-23 23:29:25,747] INFO:add_missense_variant_annotations: === Auto-downloading annotation scores ===
[2026-06-23 23:29:25,991] INFO:add_missense_variant_annotations:   Already complete: /s/project/ukbbgym/annotation_files/more_missense/tmp/revel.zip
[2026-06-23 23:29:26,044] INFO:add_missense_variant_annotations:   Already extracted: /s/project/ukbbgym/annotation_files/more_missense/tmp/revel.zip -> /s/project/ukbbgym/annotation_files/more_missense/tmp/revel
[2026-06-23 23:29:26,439] INFO:add_missense_variant_annotations:   Already present: /s/project/ukbbgym/annotation_files/more_missense/tmp/ClinPred_hg38.txt.gz
[2026-06-23 23:29:26,663] INFO:add_missense_variant_annotations:   Already present: /s/project/ukbbgym/annotation_files/more_missense/tmp/bayesdel.gz
[2026-06-23 23:29:26,664] INFO:add_missense_variant_annotations: ==

# Add clinical significance labels to the final annotation output

In [ ]:
cv_all = (
    pl.read_parquet("/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet")
)

cv_all

In [ ]:
annotation_dir = "/s/project/ukbbgym/annotation_files/clinvar"
cv_labels = pl.read_parquet(f"{annotation_dir}/clinvar_labels_20260621.parquet")
cv_labels

In [ ]:
cv_all_labs = (
    cv_labels
    .join(
        cv_all,
        on="id",
        # how="left",
        validate="1:m"
    )
)

cv_all_labs

In [ ]:
cv_all_labs.write_parquet(
    f"{annotation_dir}/clinvar_significance_vep_annotations_processed_cadd_fill_na_20260621.parquet"
)